# Volume 3 Drills — Value Function Approximation & DQN

Work through each drill problem. Code cells are provided where applicable.
Answer conceptual questions in the markdown cells below each prompt.

## Drill 1 — Linear FA: Compute V(s)

Given weight vector **w = [0.5, -0.3]** and feature vector **φ(s) = [1, 2]**, compute the estimated value **V(s) = w · φ(s)**.

Show your arithmetic.

In [ ]:
import numpy as np

w = np.array([0.5, -0.3])
phi = np.array([1, 2])

V_s = np.dot(w, phi)
print(f"V(s) = w · φ(s) = {w} · {phi} = {V_s}")
# Expected: 0.5*1 + (-0.3)*2 = 0.5 - 0.6 = -0.1

## Drill 2 — Semi-Gradient Update by Hand

Given:
- α = 0.1 (step size)
- δ = 0.4 (TD error)
- φ(s) = [1, 2] (feature vector)

The semi-gradient TD(0) weight update is:

$$\Delta w = \alpha \cdot \delta \cdot \phi(s)$$

Compute Δw and show what w becomes if it started at [0.0, 0.0].

In [ ]:
alpha = 0.1
delta = 0.4
phi = np.array([1.0, 2.0])
w = np.array([0.0, 0.0])

delta_w = alpha * delta * phi
w_new = w + delta_w
print(f"Δw = α · δ · φ = {alpha} × {delta} × {phi} = {delta_w}")
print(f"w_new = {w_new}")

## Drill 3 — Convergence of Semi-Gradient TD(0)

**Question:** Is semi-gradient TD(0) guaranteed to converge?

**Answer (fill in):** ___

<details><summary>Hint</summary>
Think about linear vs nonlinear function approximation. The deadly triad matters here.
</details>

<details><summary>Answer</summary>

- **Yes** with **linear** function approximation under standard stochastic approximation conditions (Robbins-Monro step sizes, on-policy or certain off-policy distributions).
- **No** in general with **nonlinear** function approximation (e.g., deep neural networks). Divergence is possible, especially off-policy.
</details>

## Drill 4 — DQN: Purpose of the Replay Buffer

**Question:** What is the purpose of the **experience replay buffer** in DQN?

**Answer (fill in):** ___

<details><summary>Answer</summary>

The replay buffer stores past `(s, a, r, s', done)` transitions. During training, mini-batches are sampled **randomly** from this buffer.

This serves two purposes:
1. **Breaks temporal correlations** between consecutive training samples, which would otherwise bias gradient updates.
2. **Increases data efficiency** by reusing each experience multiple times.
</details>

## Drill 5 — DQN: Purpose of the Target Network

**Question:** What is the purpose of the **target network** in DQN?

**Answer (fill in):** ___

<details><summary>Answer</summary>

The target network is a **frozen copy** of the online Q-network whose parameters θ⁻ are updated only every N steps.

Without it, the TD target `r + γ max_a Q(s', a; θ)` would shift every gradient step, creating a **moving target** problem that causes oscillation or divergence. The target network keeps TD targets **stable** between updates.
</details>

## Drill 6 — Double DQN

**Question:** How does Double DQN differ from vanilla DQN?

**Answer (fill in):** ___

<details><summary>Answer</summary>

In vanilla DQN, the same network selects **and** evaluates the greedy action:

$$y = r + \gamma Q(s', \arg\max_{a'} Q(s', a'; \theta); \theta^-)$$

This leads to **overestimation bias**. Double DQN decouples the two steps:
- **Online network** (θ) selects the action: `a* = argmax Q(s', a'; θ)`
- **Target network** (θ⁻) evaluates it: `y = r + γ Q(s', a*; θ⁻)`

This significantly reduces overestimation.
</details>

## Drill 7 — Debug: Target Network Update Frequency

**Find the bug in this DQN training loop:**

```python
for step in range(total_steps):
    action = epsilon_greedy(online_net, state)
    next_state, reward, done, _ = env.step(action)
    replay_buffer.add(state, action, reward, next_state, done)

    if len(replay_buffer) >= batch_size:
        batch = replay_buffer.sample(batch_size)
        loss = compute_dqn_loss(online_net, target_net, batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        # BUG IS HERE:
        target_net.load_state_dict(online_net.state_dict())  # Update every step

    state = next_state
```

**What is the bug? How do you fix it?**

<details><summary>Answer</summary>

**Bug:** The target network is copied from the online network **every gradient step**. This defeats its purpose — the TD target moves every step, causing instability.

**Fix:** Only update the target network every `target_update_freq` steps:

```python
if step % target_update_freq == 0:
    target_net.load_state_dict(online_net.state_dict())
```

Typical value: `target_update_freq = 1000` or `10000`.
</details>

## Drill 8 — Code: Implement a Replay Buffer

In [ ]:
from collections import deque
import random

class ReplayBuffer:
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)

    def add(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        return (
            np.array(states),
            np.array(actions),
            np.array(rewards, dtype=np.float32),
            np.array(next_states),
            np.array(dones, dtype=np.float32),
        )

    def __len__(self):
        return len(self.buffer)

# Test
buf = ReplayBuffer(capacity=1000)
for i in range(10):
    buf.add(state=[i, i], action=i % 2, reward=float(i), next_state=[i+1, i+1], done=(i == 9))

print(f"Buffer size: {len(buf)}")
states, actions, rewards, next_states, dones = buf.sample(4)
print(f"Sampled rewards: {rewards}")

## Drill 9 — Code: Epsilon Decay Schedule

In [ ]:
def epsilon_schedule(step, eps_start=1.0, eps_end=0.01, eps_decay_steps=10000):
    """Linearly decay epsilon from eps_start to eps_end over eps_decay_steps."""
    fraction = min(step / eps_decay_steps, 1.0)
    return eps_start + fraction * (eps_end - eps_start)

# Visualise the schedule
steps = [0, 1000, 5000, 10000, 15000]
for s in steps:
    eps = epsilon_schedule(s)
    print(f"Step {s:6d}: ε = {eps:.4f}")

## Drill 10 — Code: Soft Target Network Update

In [ ]:
def soft_update(target_params, online_params, tau=0.005):
    """
    Polyak averaging: θ_target = τ * θ_online + (1 - τ) * θ_target
    Commonly used in DDPG/TD3/SAC instead of hard target copy.
    """
    return [tau * op + (1.0 - tau) * tp for tp, op in zip(target_params, online_params)]

# Simulate with scalar 'params'
target = [1.0, 2.0, 3.0]  # θ_target
online = [0.0, 0.0, 0.0]  # θ_online (just reset)
tau = 0.005

updated = soft_update(target, online, tau)
print(f"After soft update (τ={tau}): {updated}")
print("(Target barely moved — slow tracking of online network)")

## Drill 11 — The Deadly Triad

**Question:** What is the **"deadly triad"** in reinforcement learning, and why is it dangerous?

**Answer (fill in):** ___

<details><summary>Answer</summary>

The deadly triad (Sutton & Barto) refers to the combination of three elements that together can cause **divergence** in RL:

1. **Function approximation** (e.g., neural networks)
2. **Bootstrapping** (using estimated values to update other estimates, as in TD learning)
3. **Off-policy learning** (training on data from a different policy than the one being evaluated)

Any two of the three are generally safe. All three together can produce instability and divergence (as seen in naive off-policy deep RL).
</details>

## Drill 12 — Trace: DQN TD Loss for a Batch

Given a batch of 4 transitions (simplified, 1D state):

| s  | a | r   | s'  | done |
|----|---|-----|-----|------|
| 0  | 0 | 1.0 | 1   | 0    |
| 1  | 1 | 0.0 | 2   | 0    |
| 2  | 0 | 0.5 | 3   | 0    |
| 3  | 1 | 2.0 | 4   | 1    |

Assume γ=0.99, and Q_target(s', a) = s'/10 for all a (dummy target net).

In [ ]:
gamma = 0.99

transitions = [
    (0, 0, 1.0, 1, 0),
    (1, 1, 0.0, 2, 0),
    (2, 0, 0.5, 3, 0),
    (3, 1, 2.0, 4, 1),
]

def dummy_Q_online(s, a):
    return s / 10.0  # pretend online network

def dummy_Q_target_max(s_next):
    return s_next / 10.0  # max over actions (same here)

print(f"{'s':>3} {'a':>3} {'r':>5} {'s\'':>4} {'done':>5} | {'Q(s,a)':>8} {'target':>10} {'TD error':>10}")
print("-" * 60)
losses = []
for s, a, r, s_next, done in transitions:
    q_sa = dummy_Q_online(s, a)
    q_next_max = dummy_Q_target_max(s_next)
    target = r + gamma * q_next_max * (1 - done)
    td_error = target - q_sa
    losses.append(td_error ** 2)
    print(f"{s:>3} {a:>3} {r:>5.1f} {s_next:>4} {done:>5} | {q_sa:>8.3f} {target:>10.3f} {td_error:>10.3f}")

print(f"\nMSE Loss = {np.mean(losses):.4f}")

## Drill 13 — Huber Loss vs MSE for Large TD Errors

**Question:** For TD error δ = 2.5, compute both MSE loss and Huber loss (δ_clip = 1.0). Which is smaller and why is that useful?

<details><summary>Answer</summary>

- **MSE:** δ² = 2.5² = **6.25**
- **Huber (δ_clip=1):** For |δ| > 1: L = δ_clip × |δ| − 0.5 × δ_clip² = 1×2.5 − 0.5 = **2.0**

Huber loss is smaller (2.0 vs 6.25). For large errors it behaves like MAE (linear), which **clips gradient magnitude** and prevents large TD errors from causing destructive weight updates.
</details>

In [ ]:
def mse_loss(td_error):
    return td_error ** 2

def huber_loss(td_error, delta=1.0):
    abs_err = abs(td_error)
    if abs_err <= delta:
        return 0.5 * td_error ** 2
    else:
        return delta * abs_err - 0.5 * delta ** 2

for err in [0.5, 1.0, 2.5, 5.0]:
    print(f"δ={err:.1f}  MSE={mse_loss(err):.3f}  Huber={huber_loss(err):.3f}")

## Drill 14 — Debug: Missing Done Flag in Target Computation

**Find the bug:**

```python
def compute_target(reward, next_q_max, gamma=0.99):
    # BUG: done flag ignored — always bootstraps!
    return reward + gamma * next_q_max
```

**What is the bug? What happens in terminal states?**

<details><summary>Answer</summary>

**Bug:** The `done` flag is never used. At terminal states (done=True), there is no future value — the episode ends. The correct target is:

```python
def compute_target(reward, next_q_max, done, gamma=0.99):
    return reward + gamma * next_q_max * (1.0 - float(done))
```

Without this fix, the agent incorrectly bootstraps past episode boundaries, thinking there's a next state even when the episode has ended. This leads to **overestimation** and slow/incorrect learning near terminal transitions.
</details>

## Drill 15 — Challenge: Prioritized Experience Replay (Conceptual)

**Question:** Describe how Prioritized Experience Replay (PER) works conceptually. What replaces uniform sampling?

**Answer (fill in):** ___

<details><summary>Answer</summary>

**Key idea:** Sample transitions that the agent can learn more from (i.e., larger TD errors) more frequently.

1. **Priority:** Each transition i is assigned priority `p_i = |δ_i| + ε` where δ_i is the TD error and ε is a small constant to ensure non-zero probability.

2. **Sampling probability:** `P(i) = p_i^α / Σ p_j^α`  (α=0: uniform, α=1: fully proportional)

3. **Importance sampling weights:** To correct for the bias introduced by non-uniform sampling:
   `w_i = (1 / (N × P(i)))^β`  (β annealed from 0.4 → 1.0)

4. **Update:** After computing TD errors on the sampled batch, update each transition's priority in the buffer.

PER can significantly improve sample efficiency and convergence speed (used in Rainbow DQN).
</details>